# 01 — Data Collection

Collect closed issues from the configured repos via the GitHub **GraphQL** API and apply the labeling rules.

All logic lives in `src/`; this notebook orchestrates and narrates. Knobs (repos, date windows, label sets) are in `config.yaml`.

**Why GraphQL:** the *linked merged PR* signal that defines Class 1 is not on the REST issues endpoint, and the REST timeline endpoint costs one request per issue. One GraphQL `search` page pulls the issue + labels + timeline (closer PR and cross-referenced PRs) together, and `rateLimit` is read on every call.

In [1]:
# --- Layout-agnostic bootstrap ------------------------------------------------
# Runs from the development repo (src/, data/processed/) OR a packaged copy
# folder (Source_Code/, Dataset/). It finds the source package, makes it
# importable as `src`, points the project root at a writable location, and stages
# config.yaml + the dataset onto the paths the code expects. Every step is a
# no-op in the dev layout, so it changes nothing there.
import os, sys, pathlib, shutil, importlib

def _bootstrap():
    here = pathlib.Path.cwd()
    root = pkg = None
    for d in [here, *here.parents]:
        for name in ('src', 'Source_Code'):
            if (d / name / 'config.py').exists():
                root, pkg = d, d / name
                break
        if root:
            break
    if root is None:
        raise RuntimeError('Could not find src/ or Source_Code/ from ' + str(here))
    os.environ['GHIC_PROJECT_ROOT'] = str(root)
    # config.py reads PROJECT_ROOT/config.yaml; stage it if it lives in the package dir.
    if not (root / 'config.yaml').exists() and (pkg / 'config.yaml').exists():
        shutil.copy(pkg / 'config.yaml', root / 'config.yaml')
    # Code reads PROJECT_ROOT/data/processed/; stage the dataset from Dataset/ if needed.
    proc = root / 'data' / 'processed'
    proc.mkdir(parents=True, exist_ok=True)
    for fname in ('combined.csv', 'releases.json'):
        if not (proc / fname).exists() and (root / 'Dataset' / fname).exists():
            shutil.copy(root / 'Dataset' / fname, proc / fname)
    # Make the package importable as `src` regardless of its folder name.
    sys.path.insert(0, str(root if pkg.name == 'src' else pkg.parent))
    if pkg.name != 'src' and 'src' not in sys.modules:
        sys.modules['src'] = importlib.import_module(pkg.name)
    return root

_bootstrap()
# ------------------------------------------------------------------------------
from ghic.config import get_config
from ghic import collect, label, utils
print('project root:', utils.PROJECT_ROOT)

project root: D:\mllab\github-issue-classifier


## Token check
Collection needs `GH_TOKEN` in `.env` (see `.env.example`). Labeling/feature/modeling notebooks do **not**.

In [2]:
# Collection needs a token; everything downstream (labeling, features, modeling)
# does not. Load with the token when available, else fall back so the rest of
# the notebook still runs from the provided combined.csv.
try:
    cfg = get_config(require_token=True)
    HAVE_TOKEN = True
    print('Token loaded. Repos:', [r.slug for r in cfg.repos])
except RuntimeError as e:
    cfg = get_config(require_token=False)
    HAVE_TOKEN = False
    print('NO TOKEN:', e)
    print('\nContinuing without collection. The cells below use the provided',
          'combined.csv, so notebooks 02 and 03 still run end to end.')

2026-06-03 01:11:03,021 | INFO    | ghic.config | Loaded config: 3 repos, 7 bot logins, 7 bug labels, 12 non-actionable labels, 6 question labels (drop=True)


NO TOKEN: GH_TOKEN is not set. Add it to .env (see .env.example) or export it in your shell before running collection.

Continuing without collection. The cells below use the provided combined.csv, so notebooks 02 and 03 still run end to end.


## Cost probe
One probe page projects the worst-case point budget before committing to a full run (5000 points/hour limit).

In [3]:
# Cost probe makes one live API call, so it only runs when a token is present.
if HAVE_TOKEN:
    collect.measure_cost(cfg)  # logs per-page cost + projected total + % of hourly budget
else:
    print('Skipped: no token. The cost probe needs a live API call.')

Skipped: no token. The cost probe needs a live API call.


## Full collection
Idempotent: cached pages under `data/raw/` are reused; the run resumes from the first uncached cursor and self-aborts at `max_runtime_minutes`. Writes per-repo CSVs plus `data/processed/combined.csv`.

Prefer running the CLI for long jobs: `python -m ghic.collect`.

In [4]:
# Full collection hits the GitHub API. Skip it when the cleaned combined.csv is
# already present (the submitted case), so this notebook is reproducible offline.
combined = utils.DATA_PROCESSED / 'combined.csv'
if combined.exists():
    print('combined.csv already present, skipping collection:', combined)
    print('To force a fresh run, delete it (or run `python -m ghic.collect`) with a token set.')
elif HAVE_TOKEN:
    result = collect.run_collection(cfg)
    print(result)
else:
    print('No combined.csv and no token. Add GH_TOKEN to .env and re-run,',
          'or use the provided combined.csv.')

combined.csv already present, skipping collection: D:\mllab\github-issue-classifier\data\processed\combined.csv
To force a fresh run, delete it (or run `python -m ghic.collect`) with a token set.


## Labeling audit
Apply the rules and print the histogram — how many issues hit each rule, the C1 ratio, and R4's share of the kept set. This histogram is the defense of the labels.

In [5]:
combined = utils.DATA_PROCESSED / 'combined.csv'
if not combined.exists():
    print('Missing', combined, '\nRun the collection cell above, or `python -m ghic.collect`.')
else:
    issues = label._read_collected_csv(combined)
    kept, audit = label.label_dataset(issues, cfg)
    print(label.format_audit(audit))
    label._write_labeled_csv(utils.DATA_PROCESSED / 'labeled.csv', kept)

Labeling audit (input=6175):
  DROP  drop_no_author                             37
  DROP  drop_bot                                    0
  DROP  drop_locked                                 0
  DROP  drop_question                             253
  C1    R1_merged_pr_link                         680
  C1    R1b_merged_pr_textual_reference            31
  C1    R2_bug_label_completed                    926
  C0    R3_non_actionable_label                   312
  C0    R3b_state_not_planned                    2102
  C0    R4_default_non_actionable                1834
  ----------------------------------------------------
  dropped=290, kept=5885 (C1=1637, C0=4248, C1 ratio=27.82%, R4 share of kept=31.16%)


2026-06-03 01:11:11,115 | INFO    | ghic.label | wrote D:\mllab\github-issue-classifier\data\processed\labeled.csv (5885 rows)
